# 🤖 Phase 2 — Week 1 & 2: What is ML + Your First Model (Linear Regression)
### Dataset: Social Media Post Engagement Predictor

---

**What you'll learn:**
- What Machine Learning actually is (no hype, just the real idea)
- Supervised vs Unsupervised learning
- Features vs Labels — the two most important ML vocabulary words
- Train/Test split — why it exists and how to do it
- Linear Regression — your first real ML model
- Model evaluation — how do you know if your model is good?

> 💡 By the end of this notebook you will have trained, evaluated, and interpreted a real ML model. Not a toy — a real one.

---
## 🧠 Section 1: What is Machine Learning — Really?

Forget the hype. Here's the real idea:

> **ML = finding patterns in data automatically, instead of writing rules manually.**

### The old way (rule-based programming):
```
IF likes > 5000 AND shares > 500 THEN post is viral
IF likes < 1000 AND shares < 100 THEN post is not viral
```
You write the rules. Works for simple cases. Breaks on complex ones.

### The ML way:
```
Here are 10,000 posts with their likes, shares, comments.
Here are the labels: which ones went viral.
Figure out the rules yourself.
```
The model learns the rules from data. No manual if/else needed.

---

### Supervised vs Unsupervised

| Type | What it means | Example |
|---|---|---|
| **Supervised** | You give data AND the correct answers | Predict engagement from likes/shares |
| **Unsupervised** | You give data only, model finds patterns | Group similar posts together |

**90% of real ML jobs are supervised learning.** That's what Phase 2 focuses on.

---

### The two most important words in ML:

- **Features (X)** — the inputs you give the model (likes, comments, shares, platform)
- **Label (y)** — the output you want to predict (total engagement, is_viral)

In your NestJS/Prisma world:
- Features = the columns you query
- Label = the column you want to predict

That's it. Everything in ML comes back to: **given X, predict y.**

---
## 📐 Section 2: Linear Regression — The Intuition

Linear Regression answers one question:
> **"If I know X, what number can I predict for y?"**

Example: Can I predict a post's total **engagement** if I know its **likes**?

The model finds the best straight line through your data:
```
engagement = (weight × likes) + bias
```

You already know this from math: **y = mx + b**
- `m` = slope (weight) — how much engagement increases per extra like
- `b` = intercept (bias) — baseline engagement even with 0 likes

ML training = **finding the best m and b automatically from your data.**
That's genuinely all Linear Regression is.

In [ ]:
# ✅ Step 1: Generate dataset and visualize the relationship

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
n = 200

df = pd.DataFrame({
    "likes":    np.random.randint(100, 20000, n),
    "comments": np.random.randint(10, 2000, n),
    "shares":   np.random.randint(5, 1500, n),
})

# Label: total engagement (what we want to predict)
df["engagement"] = df["likes"] + df["comments"] + df["shares"]

print(df.shape)
df.head()


In [ ]:
# ✅ Visualize: does likes correlate with engagement?
# This is what you ALWAYS do before training — look at the data first!

plt.figure(figsize=(8, 5))
plt.scatter(df["likes"], df["engagement"], alpha=0.5, color="#6366f1")
plt.title("Likes vs Engagement (before training)")
plt.xlabel("Likes")
plt.ylabel("Total Engagement")
plt.grid(True, alpha=0.3)
plt.show()

# If you see a diagonal pattern → linear relationship exists → Linear Regression will work!


---
## ✂️ Section 3: Train / Test Split — A Critical Concept

This is one of the most important ideas in all of ML.

**The problem:** If you train your model on ALL your data, you can't tell if it actually learned patterns or just memorized the data.

**The solution:** Split your data into two parts:
- **Training set (80%)** — the model learns from this
- **Test set (20%)** — you hide this from the model, then use it to check if the model generalizes

**Real-world analogy:** Textbook + exam.
- Training = studying the textbook
- Testing = the exam with questions the model has never seen before

If your model only scores well on training data but badly on test data → it **overfit** (memorized, didn't learn).
This concept will follow you through every single ML project.

In [ ]:
from sklearn.model_selection import train_test_split

# Define Features (X) and Label (y)
X = df[["likes", "comments", "shares"]]   # inputs — 2D table
y = df["engagement"]                       # output — 1D column

print(f"X shape: {X.shape}")   # (200, 3) — 200 rows, 3 features
print(f"y shape: {y.shape}")   # (200,) — 200 labels

# Split: 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"\nTraining rows: {X_train.shape[0]}")
print(f"Testing rows:  {X_test.shape[0]}")


---
## 🏋️ Section 4: Train Your First Model

Now the fun part. Scikit-learn (sklearn) is the standard ML library. Every model follows the same 3-step pattern:

```python
model = SomeModel()       # 1. Create
model.fit(X_train, y_train)  # 2. Train
model.predict(X_test)     # 3. Predict
```

This pattern is the same for **every** ML model you'll ever use. Learn it once, use it forever.

In [ ]:
from sklearn.linear_model import LinearRegression

# Step 1: Create the model
model = LinearRegression()

# Step 2: Train — model finds the best weights (m) and bias (b) from training data
model.fit(X_train, y_train)

print("✅ Model trained!")
print(f"\nWeights (one per feature): {model.coef_}")
print(f"  likes weight:    {model.coef_[0]:.4f}")
print(f"  comments weight: {model.coef_[1]:.4f}")
print(f"  shares weight:   {model.coef_[2]:.4f}")
print(f"\nBias (intercept): {model.intercept_:.4f}")

# 💡 Notice the weights are all close to 1.0
# That makes sense! engagement = likes + comments + shares
# The model figured that out from the data alone — no hardcoding needed!


In [ ]:
# Step 3: Make predictions on test data (data model has NEVER seen)
y_pred = model.predict(X_test)

# Compare first 10 predictions vs actual values
results = pd.DataFrame({
    "Actual":    y_test.values[:10],
    "Predicted": y_pred[:10].round(0),
    "Difference": (y_test.values[:10] - y_pred[:10]).round(0)
})
print(results.to_string(index=False))


---
## 📏 Section 5: Model Evaluation — Is It Actually Good?

Predictions look close, but how do we measure "good" formally?

**Three key metrics for regression (predicting numbers):**

| Metric | What it measures | Good value |
|---|---|---|
| **MAE** (Mean Absolute Error) | Average prediction error in original units | Lower is better |
| **RMSE** (Root Mean Squared Error) | Same, but penalizes big errors more | Lower is better |
| **R² Score** | % of variance explained by the model | Closer to 1.0 is better |

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2   = r2_score(y_test, y_pred)

print("=== MODEL EVALUATION ===")
print(f"MAE:   {mae:.1f}")
print(f"RMSE:  {rmse:.1f}")
print(f"R²:    {r2:.4f}")

print(f"\n📊 Interpretation:")
print(f"  On average, predictions are off by {mae:.0f} engagement points")
print(f"  The model explains {r2*100:.1f}% of the variation in engagement")


In [ ]:
# ✅ Visualize: Actual vs Predicted
# Perfect model = all dots on the diagonal line

plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred, alpha=0.5, color="#6366f1")

# Draw the perfect prediction line
min_val = min(y_test.min(), y_pred.min())
max_val = max(y_test.max(), y_pred.max())
plt.plot([min_val, max_val], [min_val, max_val], "r--", label="Perfect prediction")

plt.title("Actual vs Predicted Engagement")
plt.xlabel("Actual Engagement")
plt.ylabel("Predicted Engagement")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


In [ ]:
# ✅ Predict on a brand new post — this is the real use case!
# Someone creates a post. Before it goes viral, can we predict engagement?

new_post = pd.DataFrame({
    "likes":    [3500],
    "comments": [280],
    "shares":   [120]
})

predicted_engagement = model.predict(new_post)[0]
print(f"New post likes: 3500, comments: 280, shares: 120")
print(f"Predicted total engagement: {predicted_engagement:,.0f}")
print(f"Actual (manual): {3500 + 280 + 120}")


---
## 🎯 Section 6: Exercises

### Exercise 1
Train a new Linear Regression model using **only `likes`** as the feature (not all 3). Compare its R² score to the 3-feature model. What does this tell you?

### Exercise 2
Add a new feature `like_to_comment_ratio = likes / comments` to `df`. Retrain the model with all 4 features. Does R² improve?

### Exercise 3
Predict the engagement for these 3 hypothetical posts and rank them:
- Post A: likes=500, comments=50, shares=20
- Post B: likes=8000, comments=100, shares=30
- Post C: likes=2000, comments=800, shares=600

> 💡 Exercise 2 is called **feature engineering** — creating new features from existing ones. It's one of the most powerful skills in ML.

In [ ]:
# 🏋️ YOUR WORKSPACE

# ✏️ Exercise 1:


# ✏️ Exercise 2:


# ✏️ Exercise 3:



---
## ✅ Week 1–2 Checklist

- [ ] Can explain ML in plain words (no hype)
- [ ] Know the difference between supervised and unsupervised
- [ ] Know what Features (X) and Label (y) mean
- [ ] Understand why train/test split exists
- [ ] Trained a Linear Regression model using sklearn's 3-step pattern
- [ ] Can interpret MAE, RMSE, and R² scores
- [ ] Completed all 3 exercises

---

## 🔜 What's Next — Phase 2, Week 3–4

**Logistic Regression + Classification**

Instead of predicting a number, you'll predict a **category**: is a post viral or not? (yes/no)

New concepts coming:
- Classification vs Regression
- Accuracy, Precision, Recall, F1-score
- Confusion matrix
- Your project: **Spam Post Detector**

> 💬 Finish the exercises and come back — Phase 2 is just getting started!